# Assignment Beta — Question 2: Parameter-efficient fine-tuning (Google Colab)
**Student ID:** 22P-9288 · **Model:** `Qwen/Qwen2.5-1.5B-Instruct`

Complete practical experiment: **base model, LoRA r=8, QLoRA NF4 r=8, DoRA r=8, AdaLoRA init=12→target=8**, plus a rank sweep `r=4,16,32` on the winning approach.

**Before running: Runtime → Change runtime type → T4 GPU or better.** Keep Google Drive mounted; per-run metrics and predictions persist for resumable runs. The notebook installs its packages, writes the full experiment code, constructs the reproducible 2,000-example dataset and evaluates a truly held-out 200-example set.

**Important dataset disclosure:** This is a controlled **synthetic function-calling task dataset**, not the external Glaive dataset. Its exact four-tool schema matches the assignment scenario and permits a controlled same-data comparison. It should not be represented as real customer traffic. [Glaive public dataset](https://huggingface.co/datasets/glaiveai/glaive-function-calling-v2) is available but has different tools and conversation format; replacing the data changes the experiment.
**Recovery revision (14 Nov 2026):** AdaLoRA gradient-clearing bug fixed. Run the code-writer cell first, then optional summary recovery, then AdaLoRA only. The rank sweep requires GPU and three more real runs. Do not interpret reconstructed summary metrics as raw evidence.


In [ ]:
%pip install -q \
  "transformers==4.57.3" \
  "peft==0.17.1" \
  "accelerate==1.12.0" \
  "bitsandbytes==0.48.2" \
  "datasets==3.6.0" \
  "huggingface-hub==0.36.2" \
  "safetensors>=0.4"
import torch, sys
assert torch.cuda.is_available(), 'GPU required. Colab menu: Runtime → Change runtime type → T4 GPU'
print('torch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))
print('GPU memory (GiB):',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))

## 1. Use Google Drive to persist results
Automatic resume skips a completed run if its `q2_metrics.json` entry exists. If memory fails, reduce batch size **for every method** and restart all runs to preserve fairness; do not compare runs with different experimental settings.

In [ ]:
from pathlib import Path
import os
try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    ROOT=Path('/content/drive/MyDrive/Assignment_Beta_22P-9288')
except Exception as e:
    ROOT=Path('/content/Assignment_Beta_22P-9288')
    print('Drive unavailable; results are temporary:',e)
for name in ['code','data','results','figures']:(ROOT/name).mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
print('Working folder:',ROOT)

## 2. Build exactly 2,000 labelled samples
- 500 per tool: `check_order_status`, `initiate_refund`, `track_shipment`, `escalate_to_human`.
- Deterministic split: 1,800 train + 200 held-out test, 50 test per class.
- Train/test identities are disjoint. The test set is not used for training or prompt selection.
- The synthetic template approach is a limitation; discuss generalization honestly.

In [ ]:
from pathlib import Path
Path('code/prepare_q2_data.py').parent.mkdir(parents=True,exist_ok=True)
Path('code/prepare_q2_data.py').write_text('#!/usr/bin/env python3\n"""Construct 2000 **synthetic** labelled support conversations, with sealed test split.\n\nThis transparent, controlled 4-function task dataset is allowed by the prompt\'s\n\'such as glaive-function-calling-v2\' wording, but is NOT Glaive. Because it is\nsynthetic and template-based, final claims require validation on human queries.\n"""\nimport json,random\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[1];OUT=ROOT/\'data\';OUT.mkdir(exist_ok=True)\nSEED=2026; rng=random.Random(SEED)\nTEMPLATES={\n \'check_order_status\':[\n  \'Could you check the status of order {oid}?\', \'Is order {oid} processing or confirmed?\',\n  \'What is the current state of my purchase {oid}?\', \'My order {oid} was placed on {day}; what is its status?\',\n  \'Please check if {oid} was accepted.\', \'Order {oid}: has it been processed yet?\',\n  \'I want an update about order {oid}.\' ],\n \'track_shipment\':[\n  \'Where is the parcel with tracking number {tid}?\', \'Track shipment {tid} please.\',\n  \'Can you locate my package {tid}?\', \'What is the delivery ETA for tracking {tid}?\',\n  \'Shipment {tid} has not arrived. Where is it?\', \'Please show the courier progress for {tid}.\',\n  \'Has tracking {tid} reached the local sorting facility?\' ],\n \'initiate_refund\':[\n  \'I want a refund for order {oid} because {reason}.\', \'Please start a refund for order {oid}: {reason}.\',\n  \'Refund order {oid}, {reason}.\', \'I explicitly request a refund for {oid} due to {reason}.\',\n  \'Can you begin my refund request for {oid}? Reason: {reason}.\',\n  \'I returned order {oid} and request a refund as {reason}.\',\n  \'Please open a refund claim for {oid}; {reason}.\' ],\n \'escalate_to_human\':[\n  \'Please connect me with a human agent about {issue}.\',\n  \'I need a real support representative; {issue}.\',\n  \'I refuse the automated support, please escalate: {issue}.\',\n  \'A supervisor needs to review my case: {issue}.\',\n  \'I cannot provide my order ID and I need help: {issue}.\',\n  \'Connect me with customer care, {issue}.\',\n  \'The chatbot is not resolving this. Human help required: {issue}.\' ]}\nDAYS=[\'Monday\',\'Tuesday\',\'Wednesday\',\'Thursday\',\'Friday\',\'Saturday\']\nREASONS=[\'the item arrived damaged\',\'the product was defective\',\'I received the wrong size\',\n         \'the item was missing from the box\',\'the order was duplicated\',\'it arrived incomplete\']\nISSUES=[\'a disputed charge\',\'account access\',\'a missing order number\',\'multiple failed refunds\',\n        \'a sensitive complaint\',\'a payment dispute\',\'a lost confirmation email\']\nrows=[]\nfor category in TEMPLATES:\n    for idx in range(500):\n        # Every example has unique identifiers; categories remain balanced.\n        oid=f\'ORD-{idx+1:05d}-{category[:2].upper()}\'\n        tid=f\'TRK-{idx+1:06d}-{category[:2].upper()}\'\n        reason=REASONS[(idx*3 + idx//9)%len(REASONS)]\n        issue=ISSUES[(idx*5 + idx//11)%len(ISSUES)]\n        template=TEMPLATES[category][(idx*11 + idx//7)%len(TEMPLATES[category])]\n        message=template.format(oid=oid,tid=tid,reason=reason,issue=issue,day=DAYS[idx%len(DAYS)])\n        if category==\'check_order_status\':arguments={\'order_id\':oid}\n        elif category==\'track_shipment\':arguments={\'tracking_number\':tid}\n        elif category==\'initiate_refund\':arguments={\'order_id\':oid,\'reason\':reason}\n        else:arguments={\'issue\':issue}\n        rows.append({\'id\':f\'{category}_{idx:04d}\',\'message\':message,\n                     \'answer\':{\'name\':category,\'arguments\':arguments}})\n# Make exactly 50 unseen examples per class. This split is fixed once and\n# never consulted by training/early stopping or hyperparameter selection.\ntrain=[];test=[]\nfor category in TEMPLATES:\n    block=[row for row in rows if row[\'answer\'][\'name\']==category]\n    rng.shuffle(block);test+=block[:50];train+=block[50:]\nrng.shuffle(train);rng.shuffle(test)\nfor name,items in [(\'q2_train.jsonl\',train),(\'q2_test_sealed.jsonl\',test)]:\n    p=OUT/name\n    if p.exists():\n        old=[json.loads(x) for x in p.read_text(encoding=\'utf-8\').splitlines() if x.strip()]\n        if old!=items:raise SystemExit(f\'{p} already exists and differs. Keep held-out split fixed; do not overwrite.\')\n    else:\n        with p.open(\'w\',encoding=\'utf-8\') as f:\n            for row in items:f.write(json.dumps(row,ensure_ascii=False)+\'\\n\')\nassert len(train)==1800 and len(test)==200\nassert not set(r[\'id\'] for r in train)&set(r[\'id\'] for r in test)\nprint(\'Saved/verified 1800 training and 200 sealed test examples; synthetic, 4 labels; seed\',SEED)\n', encoding="utf-8")
print("Saved", 'code/prepare_q2_data.py')

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,'code/prepare_q2_data.py'],check=True)
import json
from collections import Counter
train=[json.loads(line) for line in Path('data/q2_train.jsonl').read_text().splitlines()]
test=[json.loads(line) for line in Path('data/q2_test_sealed.jsonl').read_text().splitlines()]
print('Train:',len(train),'Test:',len(test))
print('Training labels:',Counter(r['answer']['name'] for r in train))
print('Test labels:',Counter(r['answer']['name'] for r in test))
assert len(train)==1800 and len(test)==200
assert not ({x['id'] for x in train}&{x['id'] for x in test})

## 3. Write the complete training/evaluation program
Every run uses the same base model, train/test split, seed, 2 epochs, target modules (`q_proj`,`v_proj`), optimizer and LR schedule; **QLoRA alone changes weight precision (NF4), which is inherent to that method**.

The code measures peak GPU memory, training time, JSON validity, tool accuracy, and exact argument match, and stores all 200 predictions per run. It also computes 5 failure examples from real predictions. **A trained adapter never directly executes a refund.**

In [ ]:
from pathlib import Path
Path('code/q2_finetune_experiments.py').parent.mkdir(parents=True,exist_ok=True)
Path('code/q2_finetune_experiments.py').write_text('#!/usr/bin/env python3\n"""Question 2: measured baseline, LoRA, QLoRA, DoRA, AdaLoRA + rank sweep.\n\nPrerequisites: Python 3.10+, CUDA GPU, pytorch CUDA build, and:\n pip install \'transformers>=4.45,<5\' \'peft>=0.15,<0.19\' \'accelerate>=1,<2\' \\\n             \'bitsandbytes>=0.45\' \'safetensors>=0.4\' numpy matplotlib\n\nRun from the package root:\n python code/prepare_q2_data.py\n python code/q2_finetune_experiments.py --run all\n python code/q2_finetune_experiments.py --run rank-sweep\n\nThe *same* 1800 train, 200 sealed test, model, modules, epochs, seed, AdamW,\nlinear LR schedule, sequence length and optimization batch size are used for\nall methods. The one intentional variation is QLoRA\'s NF4 weight precision;\nDoRA and AdaLoRA add their method-specific parameters. Paged optimizer is NOT\nenabled, because changing optimizer only for QLoRA would confound comparison.\n\nGPU runs can take substantial time. No measured results exist until this\nscript writes results/q2_metrics.json and per-example predictions files.\n"""\nimport argparse, csv, gc, json, math, os, random, time\nfrom pathlib import Path\nimport numpy as np\nROOT=Path(__file__).resolve().parents[1]\nDATA=ROOT/\'data\';RESULTS=ROOT/\'results\';RESULTS.mkdir(exist_ok=True)\nMODEL_ID=\'Qwen/Qwen2.5-1.5B-Instruct\'\nSEED=2026\nSYSTEM=(\'You are a customer-support function-call router. Respond with EXACTLY one \'\n        \'JSON object, with keys "name" and "arguments"; no markdown or commentary. \'\n        \'Valid names: check_order_status, initiate_refund, track_shipment, escalate_to_human. \'\n        \'check_order_status arguments: {"order_id": string}; \'\n        \'initiate_refund arguments: {"order_id": string, "reason": string}; \'\n        \'track_shipment arguments: {"tracking_number": string}; \'\n        \'escalate_to_human arguments: {"issue": string}. \'\n        \'Never initiate a refund unless the customer explicitly requests one; \'\n        \'backend refund execution always requires separate authorization.\')\n\ndef load_split(filename):\n    return [json.loads(line) for line in (DATA/filename).read_text(encoding=\'utf-8\').splitlines() if line.strip()]\n\ndef seed_everything(torch):\n    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)\n\ndef prompt_text(tokenizer,message):\n    msgs=[{\'role\':\'system\',\'content\':SYSTEM},{\'role\':\'user\',\'content\':message}]\n    return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)\n\ndef make_dataset(torch,tokenizer,rows,maxlen):\n    class TrainDataset(torch.utils.data.Dataset):\n        def __len__(self):return len(rows)\n        def __getitem__(self,i):\n            row=rows[i]\n            prompt=tokenizer(prompt_text(tokenizer,row[\'message\']),add_special_tokens=False).input_ids\n            answer=json.dumps(row[\'answer\'],separators=(\',\',\':\'),ensure_ascii=False)\n            ans=tokenizer(answer,add_special_tokens=False).input_ids+[tokenizer.eos_token_id]\n            if len(prompt)+len(ans)>maxlen:\n                raise ValueError(f\'example {row["id"]} length {len(prompt)+len(ans)} > maxlen {maxlen}; do not silently truncate answers\')\n            ids=prompt+ans\n            return {\'input_ids\':ids,\'labels\':[-100]*len(prompt)+ans,\'attention_mask\':[1]*len(ids)}\n    return TrainDataset()\n\ndef collate(torch,tokenizer,batch):\n    longest=max(len(row[\'input_ids\']) for row in batch)\n    items={\'input_ids\':[], \'labels\':[], \'attention_mask\':[]}\n    for row in batch:\n        missing=longest-len(row[\'input_ids\'])\n        items[\'input_ids\'].append(row[\'input_ids\']+[tokenizer.pad_token_id]*missing)\n        items[\'labels\'].append(row[\'labels\']+[-100]*missing)\n        items[\'attention_mask\'].append(row[\'attention_mask\']+[0]*missing)\n    return {key:torch.tensor(v,dtype=torch.long,device=\'cuda\') for key,v in items.items()}\n\ndef train_method(torch,model,tokenizer,train,method,rank,epochs,batch_size,grad_accum,lr,maxlen):\n    from peft import LoraConfig,AdaLoraConfig,get_peft_model\n    from torch.utils.data import DataLoader\n    data=make_dataset(torch,tokenizer,train,maxlen)\n    loader=DataLoader(data,batch_size=batch_size,shuffle=True,generator=torch.Generator().manual_seed(SEED),\n                      collate_fn=lambda batch:collate(torch,tokenizer,batch))\n    steps_per_epoch=math.ceil(len(loader)/grad_accum);total_steps=steps_per_epoch*epochs\n    modules=[\'q_proj\',\'v_proj\']\n    if method==\'adalora\':\n        cfg=AdaLoraConfig(task_type=\'CAUSAL_LM\',init_r=max(12,int(rank*1.5)),target_r=rank,lora_alpha=16,\n            tinit=max(1,int(.1*total_steps)),tfinal=max(1,int(.1*total_steps)),\n            deltaT=10,total_step=total_steps,target_modules=modules)\n    else:\n        cfg=LoraConfig(task_type=\'CAUSAL_LM\',r=rank,lora_alpha=16,\n             lora_dropout=0,target_modules=modules,use_dora=(method==\'dora\'))\n    model=get_peft_model(model,cfg)\n    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total=sum(p.numel() for p in model.parameters())\n    model.train();model.config.use_cache=False\n    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=0.0)\n    warmup=max(1,round(.1*total_steps))\n    def schedule(step):\n        if step < warmup:return (step+1)/warmup\n        return max(0.0,(total_steps-step)/max(1,total_steps-warmup))\n    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=schedule)\n    optimizer.zero_grad(set_to_none=True)\n    torch.cuda.reset_peak_memory_stats()\n    start=time.perf_counter();step=0;running=0.\n    for epoch in range(epochs):\n        for ix,batch in enumerate(loader):\n            loss=model(**batch).loss\n            (loss/grad_accum).backward()\n            running+=float(loss.item())\n            if (ix+1)%grad_accum==0 or ix==len(loader)-1:\n                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)\n                optimizer.step();scheduler.step()\n                step+=1\n                if method==\'adalora\':\n                    # IMPORTANT: update rank importance while parameter gradients still exist.\n                    # Calling zero_grad(set_to_none=True) first makes p.grad None.\n                    model.base_model.update_and_allocate(step)\n                optimizer.zero_grad(set_to_none=True)\n                if step % 40==0:print(f\'  {method} rank={rank} epoch={epoch+1}/{epochs} step={step}/{total_steps} loss={loss.detach().item():.4f}\',flush=True)\n    torch.cuda.synchronize()\n    seconds=time.perf_counter()-start\n    peak_gb=torch.cuda.max_memory_allocated()/2**30\n    model.eval();model.config.use_cache=True\n    return model,{\'trainable_params\':trainable,\'total_params\':total,\'trainable_pct\':100*trainable/total,\n       \'peak_training_gpu_gib\':round(peak_gb,3),\'training_minutes\':round(seconds/60,3),\n       \'training_steps\':step,\'epochs\':epochs,\'rank\':rank,\'method\':method,\n       \'final_train_batch_loss\':round(float(loss.item()),5)}\n\ndef evaluate(torch,model,tokenizer,test,tag,maxlen=110):\n    rows=[];valid=0;tool_correct=0;full_correct=0;inference_seconds=[]\n    model.eval();model.config.use_cache=True\n    for i,row in enumerate(test):\n        p=tokenizer(prompt_text(tokenizer,row[\'message\']),return_tensors=\'pt\',add_special_tokens=False).to(\'cuda\')\n        torch.cuda.synchronize(); t_infer=time.perf_counter()\n        with torch.no_grad():\n            generation=model.generate(**p,max_new_tokens=maxlen,do_sample=False,num_beams=1,\n                     pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)\n        torch.cuda.synchronize(); inference_seconds.append(time.perf_counter()-t_infer)\n        raw=tokenizer.decode(generation[0,p[\'input_ids\'].shape[1]:],skip_special_tokens=True).strip()\n        obj=None;reason=\'\'\n        try:\n            obj=json.loads(raw)\n            if not isinstance(obj,dict) or not isinstance(obj.get(\'arguments\'),dict) or not isinstance(obj.get(\'name\'),str):\n                raise ValueError(\'not expected JSON object shape\')\n            valid+=1\n        except (json.JSONDecodeError,ValueError) as exc:reason=\'broken_json_or_schema\'\n        is_tool=(obj is not None and obj.get(\'name\')==row[\'answer\'][\'name\'])\n        is_full=(is_tool and obj.get(\'arguments\')==row[\'answer\'][\'arguments\'] and set(obj)=={\'name\',\'arguments\'})\n        if is_tool:tool_correct+=1\n        if is_full:full_correct+=1\n        if not reason and not is_tool:reason=\'wrong_tool\'\n        if not reason and is_tool and not is_full:reason=\'wrong_or_extra_arguments\'\n        rows.append({\'id\':row[\'id\'],\'message\':row[\'message\'],\'expected\':row[\'answer\'],\'raw_output\':raw,\n                     \'parsed\':obj,\'valid\':obj is not None,\'tool_correct\':bool(is_tool),\n                     \'full_match\':bool(is_full),\'error_category\':reason})\n        if (i+1)%25==0:print(\'  evaluation\',tag,i+1,\'/\',len(test),flush=True)\n    path=RESULTS/f\'q2_predictions_{tag}.jsonl\'\n    with path.open(\'w\',encoding=\'utf-8\') as f:\n        for row in rows:f.write(json.dumps(row,ensure_ascii=False)+\'\\n\')\n    return {\'json_valid_n\':valid,\'json_valid_rate\':valid/len(test),\n            \'tool_correct_n\':tool_correct,\'tool_accuracy\':tool_correct/len(test),\n            \'full_match_n\':full_correct,\'full_match_rate\':full_correct/len(test),\n            \'evaluation_n\':len(test),\'mean_inference_seconds\':sum(inference_seconds)/len(inference_seconds),\'predictions_file\':str(path.relative_to(ROOT))}\n\ndef run(method,rank,args,train,test,metrics):\n    import torch\n    from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig\n    from peft import prepare_model_for_kbit_training\n    seed_everything(torch);torch.cuda.empty_cache()\n    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=False)\n    if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token\n    if method==\'qlora\':\n        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n        quant_cfg=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',\n                 bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)\n        model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map={\'\':\'cuda:0\'},\n                 quantization_config=quant_cfg,trust_remote_code=False)\n        model=prepare_model_for_kbit_training(model,use_gradient_checkpointing=True)\n    else:\n        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n        model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map={\'\':\'cuda:0\'},\n                   torch_dtype=dtype,trust_remote_code=False)\n        if method!=\'base\':\n            model.gradient_checkpointing_enable()\n            model.enable_input_require_grads()\n    if method==\'base\':\n        info={\'method\':\'base\',\'rank\':0,\'trainable_params\':0,\'total_params\':sum(p.numel() for p in model.parameters()),\n              \'trainable_pct\':0,\'peak_training_gpu_gib\':None,\'training_minutes\':0,\'training_steps\':0,\'epochs\':0}\n    else:\n        model,info=train_method(torch,model,tokenizer,train,method,rank,args.epochs,args.batch_size,\n              args.grad_accum,args.learning_rate,args.max_length)\n    tag=\'base\' if method==\'base\' else f\'{method}_r{rank}\'\n    info.update(evaluate(torch,model,tokenizer,test,tag))\n    metrics[tag]=info\n    (RESULTS/\'q2_metrics.json\').write_text(json.dumps(metrics,indent=2),encoding=\'utf-8\')\n    del model;gc.collect();torch.cuda.empty_cache()\n    print(\'RESULT\',tag,json.dumps(info,indent=2),flush=True)\n\ndef plot_metrics(metrics):\n    try:\n        import matplotlib\n        matplotlib.use(\'Agg\')\n        import matplotlib.pyplot as plt\n        ranks=[(int(x.split(\'_r\')[-1]),v[\'tool_accuracy\']) for x,v in metrics.items() if \'_r\' in x and x.startswith(metrics.get(\'rank_sweep_method\',\'!\')+\'_r\')]\n        if len(ranks)>=4:\n            ranks.sort();x,y=zip(*ranks)\n            plt.figure(figsize=(7,4.5));plt.plot(x,y,\'o-\',linewidth=2)\n            plt.xlabel(\'Adapter rank\');plt.ylabel(\'Held-out tool-call accuracy\')\n            plt.title(\'Q2 rank-sensitivity (measured)\');plt.xticks(x);plt.ylim(0,1);plt.grid(alpha=.25)\n            plt.tight_layout();plt.savefig(ROOT/\'figures\'/\'q2_rank_accuracy_measured.png\',dpi=180);plt.close()\n    except ImportError:print(\'matplotlib not installed, rank data still in q2_metrics.json\')\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--run\',choices=[\'all\',\'rank-sweep\',\'base\',\'lora\',\'qlora\',\'dora\',\'adalora\'],default=\'all\')\n    parser.add_argument(\'--epochs\',type=int,default=2)\n    parser.add_argument(\'--batch-size\',type=int,default=2)\n    parser.add_argument(\'--grad-accum\',type=int,default=8)\n    parser.add_argument(\'--learning-rate\',type=float,default=2e-4)\n    parser.add_argument(\'--max-length\',type=int,default=256)\n    parser.add_argument(\'--check-only\',action=\'store_true\')\n    args=parser.parse_args()\n    train=load_split(\'q2_train.jsonl\');test=load_split(\'q2_test_sealed.jsonl\')\n    assert len(train)==1800 and len(test)==200\n    assert not {x[\'id\'] for x in train}&{x[\'id\'] for x in test}\n    if args.check_only:\n        print(\'Dataset validated (1800 train, 200 sealed test); imports and GPU not tested\');return\n    import torch\n    if not torch.cuda.is_available():raise SystemExit(\'CUDA GPU required for the specified NF4 comparison; no fake results will be generated\')\n    metrics_path=RESULTS/\'q2_metrics.json\'\n    metrics=json.loads(metrics_path.read_text()) if metrics_path.exists() else {}\n    if args.run==\'all\':\n        for method in [\'base\',\'lora\',\'qlora\',\'dora\',\'adalora\']:\n            tag=\'base\' if method==\'base\' else f\'{method}_r8\'\n            if tag in metrics:print(\'Already complete, skipping\',tag);continue\n            run(method,8,args,train,test,metrics)\n    elif args.run==\'rank-sweep\':\n        candidates=[(method,metrics[f\'{method}_r8\']) for method in [\'lora\',\'qlora\',\'dora\',\'adalora\'] if f\'{method}_r8\' in metrics]\n        if len(candidates)!=4:raise SystemExit(\'Run --run all first: results from all four methods are required\')\n        # Assignment explicitly asks to choose best after Part B; this uses the\n        # same 200 test items for selection and rank plot, causing selection bias.\n        best=max(candidates,key=lambda t:(t[1][\'full_match_rate\'],t[1][\'tool_accuracy\']))[0]\n        metrics[\'rank_sweep_method\']=best\n        for r in [4,16,32]:\n            tag=f\'{best}_r{r}\'\n            if tag not in metrics:run(best,r,args,train,test,metrics)\n        (RESULTS/\'q2_metrics.json\').write_text(json.dumps(metrics,indent=2),encoding=\'utf-8\')\n        plot_metrics(metrics)\n    else:\n        method=args.run;rank=8;run(method,rank,args,train,test,metrics)\n    print(\'Measured files:\',RESULTS)\n\nif __name__==\'__main__\':main()\n', encoding="utf-8")
print('Fixed training script saved')


In [ ]:
# Dataset/code sanity check before expensive GPU work
subprocess.run([sys.executable,'code/q2_finetune_experiments.py','--check-only'],check=True)

## 4. Run baseline + four methods (actual GPU experiment)
**Long-running step:** the model must be downloaded and training executed; no score is filled in beforehand. To make Colab sessions manageable, run one method at a time, keeping **the same arguments for every run**. The `all` option runs the baseline and four methods, resuming past completed methods.

### Resume completed experiments safely
The previous run completed baseline, LoRA r8, QLoRA r8 and DoRA r8. AdaLoRA failed due to clearing gradients too early. The patched script fixes that. **Restore `results/q2_metrics.json` from Google Drive if possible.** The optional cell below reconstructs only the four summary metrics from your console output if the original file is gone; it does not reconstruct raw predictions or model checkpoints.


In [ ]:
# OPTIONAL RECOVERY ONLY IF YOUR ORIGINAL results/q2_metrics.json IS MISSING.
# These four entries TRANSCRIBE THE EXPERIMENT RESULTS YOU POSTED IN CHAT.
# They are NOT newly rerun measurements; the raw predictions/checkpoints are NOT reconstructed.
# If your original q2_metrics.json exists, this cell preserves it.
from pathlib import Path
import json
metrics_file = Path('results/q2_metrics.json')
metrics_file.parent.mkdir(parents=True, exist_ok=True)
from_log = {
 'base': {'method':'base','rank':0,'trainable_params':0,'total_params':1543714304,'trainable_pct':0,'peak_training_gpu_gib':None,'training_minutes':0,'training_steps':0,'epochs':0,'json_valid_n':200,'json_valid_rate':1.0,'tool_correct_n':186,'tool_accuracy':.93,'full_match_n':120,'full_match_rate':.60,'evaluation_n':200,'mean_inference_seconds':1.756481384714998,'predictions_file':'results/q2_predictions_base.jsonl'},
 'lora_r8': {'method':'lora','rank':8,'trainable_params':1089536,'total_params':1544803840,'trainable_pct':.07052908413277896,'peak_training_gpu_gib':3.588,'training_minutes':46.457,'training_steps':226,'epochs':2,'json_valid_n':200,'json_valid_rate':1.,'tool_correct_n':200,'tool_accuracy':1.,'full_match_n':200,'full_match_rate':1.,'evaluation_n':200,'mean_inference_seconds':1.4719220432099882,'final_train_batch_loss':.00013,'predictions_file':'results/q2_predictions_lora_r8.jsonl'},
 'qlora_r8': {'method':'qlora','rank':8,'trainable_params':1089536,'total_params':889705984,'trainable_pct':.12246023063727084,'peak_training_gpu_gib':2.253,'training_minutes':50.599,'training_steps':226,'epochs':2,'json_valid_n':200,'json_valid_rate':1.,'tool_correct_n':200,'tool_accuracy':1.,'full_match_n':200,'full_match_rate':1.,'evaluation_n':200,'mean_inference_seconds':2.305411531730042,'final_train_batch_loss':.00014,'predictions_file':'results/q2_predictions_qlora_r8.jsonl'},
 'dora_r8': {'method':'dora','rank':8,'trainable_params':1139712,'total_params':1544854016,'trainable_pct':.07377473781962839,'peak_training_gpu_gib':3.59,'training_minutes':48.053,'training_steps':226,'epochs':2,'json_valid_n':200,'json_valid_rate':1.,'tool_correct_n':200,'tool_accuracy':1.,'full_match_n':200,'full_match_rate':1.,'evaluation_n':200,'mean_inference_seconds':1.9500327753200417,'final_train_batch_loss':.00015,'predictions_file':'results/q2_predictions_dora_r8.jsonl'}
}
if metrics_file.exists():
    metrics = json.loads(metrics_file.read_text())
    missing = [k for k in from_log if k not in metrics]
    print('Existing metrics preserved:', sorted(metrics))
    print('Missing recorded runs:', missing)
    # Do not change existing evidence without explicit review.
else:
    for k, v in from_log.items():
        v['provenance'] = 'RECOVERED_FROM_USER_SHARED_CONSOLE_OUTPUT; original raw results unavailable'
    metrics_file.write_text(json.dumps(from_log, indent=2))
    print('Rebuilt metrics SUMMARY only from supplied run log; predictions/checkpoints remain missing.')


In [ ]:
import os
import sys
import subprocess

script = "code/q2_finetune_experiments.py"

print("Working directory:", os.getcwd())
print("Script exists:", os.path.isfile(script))

if not os.path.isfile(script):
    raise FileNotFoundError(
        f"Cannot find {script}. Restore your assignment files first."
    )

cmd = [
    sys.executable, "-u", script,
    "--run", "adalora",
    "--epochs", "2",
    "--batch-size", "2",
    "--grad-accum", "8",
    "--learning-rate", "0.0002",
    "--max-length", "256"
]

print("\nLaunching:", " ".join(cmd), flush=True)

with open("q2_training_adalora.log", "w") as log:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\nExit code:", exit_code)
print("Log saved to: q2_training_adalora.log")

## 5. View five-row main comparison
You should see `base`, `lora_r8`, `qlora_r8`, `dora_r8`, `adalora_r8` **only once their evaluations have actually finished**. Files `results/q2_predictions_<method>.jsonl` retain all test examples and model outputs.

In [ ]:
import pandas as pd, json
p=Path('results/q2_metrics.json')
if p.exists():
    results=json.loads(p.read_text())
    names=['base','lora_r8','qlora_r8','dora_r8','adalora_r8']
    frame=pd.DataFrame([{'run':name,**results[name]} for name in names if name in results])
    display(frame[['run','trainable_params','trainable_pct','peak_training_gpu_gib','training_minutes','json_valid_n','tool_correct_n','full_match_n','mean_inference_seconds']] if len(frame) else frame)
else:print('No experiment results yet.')

## 6. Rank sweep (after all five main runs)
The script selects the method with highest full-match, then tool accuracy (same sealed test set per assignment) and trains at ranks **4, 16, 32**. Since the held-out set influences method selection, the final report should acknowledge **selection bias** and ideally use a separate validation split for choosing hyperparameters.

In [ ]:
cmd=[sys.executable,'code/q2_finetune_experiments.py','--run','rank-sweep',
     '--epochs',str(EPOCHS),'--batch-size',str(BATCH_SIZE),
     '--grad-accum',str(GRAD_ACCUM),'--learning-rate',str(LEARNING_RATE),
     '--max-length',str(MAX_LENGTH)]
subprocess.run(cmd,check=True)

In [ ]:
# Report the actual failed held-out examples for the selected method
from pathlib import Path
import json
p=Path('results/q2_metrics.json')
if p.exists():
    res=json.loads(p.read_text()); winner=res.get('rank_sweep_method')
    if winner:
        f=Path(f'results/q2_predictions_{winner}_r8.jsonl')
        if f.exists():
            rows=[json.loads(line) for line in f.read_text().splitlines()]
            wrong=[r for r in rows if not r['full_match']]
            print('Selected:',winner,'Wrong examples:',len(wrong))
            for x in wrong[:5]:
                print('\nCUSTOMER:',x['message'],'\nEXPECTED:',x['expected'],'\nPREDICTED:',x['raw_output'],'\nERROR:',x['error_category'])
            Path('results/q2_five_failures.json').write_text(json.dumps(wrong[:5],indent=2))
    else: print('Complete rank sweep first.')

## 7. Download evidence
Your submission should include the notebook, the full 2,000-example dataset split, per-run raw predictions, measured metrics, five failed examples and rank-accuracy plot. A `.py` source file is included for reproducibility, but the assignment names the **notebook** explicitly.

In [ ]:
import zipfile
with zipfile.ZipFile('Q2_measured_evidence.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['data','results','figures','code']:
        for file in Path(folder).glob('*'):
            if file.is_file() and (folder!='results' or file.name.startswith('q2_')) and (folder!='figures' or file.name.startswith('q2_')) and (folder!='code' or file.name in ['q2_finetune_experiments.py','prepare_q2_data.py']):
                z.write(file)
print('Created',ROOT/'Q2_measured_evidence.zip')

### Primary references
- [Hugging Face PEFT AdaLoRA](https://huggingface.co/docs/peft/main/package_reference/adalora) (rank allocation must be updated during training)
- [Hugging Face PEFT](https://huggingface.co/docs/peft/index)
- [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
- [Glaive Function Calling v2](https://huggingface.co/datasets/glaiveai/glaive-function-calling-v2) (alternative, different tool schema)

**Caveats:** Colab free GPU availability and memory are variable. QLoRA uses NF4 and double quantization. The chosen training script keeps AdamW for all runs for a controlled comparison; paged optimizers can be discussed as a QLoRA innovation without adding a second confounding change.